Based on the steps.ipynb and explainable-ai-for-tabular-dataset-xgb-with-lime.ipynb notebooks, here's a comprehensive list of features we need to build for the full Streamlit application. I've organized them into logical sections (pages/modules) for step-by-step implementation, drawing directly from the notebooks' descriptions and the existing model.py code. Each feature includes a brief outline of how to implement it, including key code snippets, dependencies, and potential pitfalls to avoid getting stuck.

The app will have three main pages (via sidebar navigation): **Overview**, **Prediction**, and **Explainability**. We'll use the pre-built functions in model.py (e.g., `build_pipeline`, `predict_customer`, SHAP/LIME explainers) to avoid reinventing the wheel. The goal is to build incrementally: start with a basic skeleton, add one page at a time, test each addition, and iterate.

### 1. **App Skeleton & Navigation (Foundation)**
   - **Feature**: Basic Streamlit setup with sidebar navigation, page routing, and cached pipeline loading.
   - **Why**: This is the backbone—ensures the app runs without errors and allows switching between pages.
   - **Outline**:
     - Use `st.set_page_config` for title, icon, and wide layout.
     - Add a sidebar with `st.sidebar.radio` for page selection ('Overview', 'Prediction', 'Explainability').
     - Cache the pipeline with `@st.cache_data` to load the model once (avoids reloading on every interaction).
     - In `main()`, route to page-specific functions (e.g., `if page == 'Overview': render_overview(raw)`).
     - Dependencies: Streamlit, pandas, plotly.
     - Pitfalls: Ensure model.py is imported correctly; test caching by running the app and checking for errors.
     - Step: Implement this first, then run `streamlit run app.py` to verify the skeleton loads.

### 2. **Overview Dashboard Page**
   - **Feature**: High-level metrics and visualizations for churn insights (e.g., total customers, churn rate, charts).
   - **Why**: Matches the "Overview Dashboard" in steps.ipynb—provides business context before diving into predictions.
   - **Sub-features**:
     - **Metrics Row**: 4 columns with metrics (total customers, churn rate, avg salary, high-risk customers). Use `st.columns` and `st.metric`.
     - **Churn Distribution Bar Chart**: Bar plot of stayed vs. churned (using `plotly.express.bar`).
     - **Churn by Geography Histogram**: Grouped bars by geography (France/Germany/Spain).
     - **Tenure and Churn Histogram**: Overlay histogram for tenure distribution by churn status.
     - **Estimated Salary vs Churn Box Plot**: Box plot comparing salaries for churned vs. stayed.
     - **Feature Correlation Heatmap**: Correlation matrix for features + 'Exited' (using `plotly.express.imshow`).
   - **Outline**:
     - Load raw data from pipeline.
     - Compute metrics (e.g., churn rate as `raw['Exited'].value_counts()[1] / len(raw)`).
     - Use Plotly for charts (e.g., `px.bar(churn_counts.reset_index(), x='index', y='Exited')`).
     - Display in a function `render_overview(raw: pd.DataFrame)`.
     - Dependencies: Plotly for charts.
     - Pitfalls: Ensure data types are correct (e.g., 'Exited' as int); test charts render without errors by adding sample data.
     - Step: Add this page after the skeleton; run and verify charts load.

### 3. **Customer Prediction Page**
   - **Feature**: Interactive form for user input, prediction output, recommendations, and explainability (SHAP/LIME).
   - **Why**: Core feature from steps.ipynb—"Customer Prediction Page (MOST IMPORTANT)"—allows ad-hoc scoring.
   - **Sub-features**:
     - **Input Form**: Two-column form with inputs (credit score, geography, gender, age, tenure, balance, products, has_card, active_member, salary). Use `st.form` and `st.form_submit_button`.
     - **Prediction Result**: Display probability, risk level (Low/Medium/High), outcome (Churn/Stay), and color-coded alerts (green/yellow/red via `st.success`/`st.warning`/`st.error`).
     - **Recommendations**: List of actions based on probability (e.g., "Offer a loyalty discount").
     - **Download Report**: CSV download of prediction details (inputs + results) using `st.download_button`.
     - **Explainability Expander**: Toggle for SHAP/LIME details.
       - **Local SHAP Contributions**: Table and bar chart of SHAP values for the prediction (top 10 features).
       - **LIME Local Explanation**: Table of feature contributions (top 5).
       - **Top Local Drivers**: Natural language summary (e.g., "Age increases churn risk by 0.123").
   - **Outline**:
     - In `render_prediction(raw, pipeline)`, use `encode_features` and `predict_customer` for scoring.
     - Compute risk/advice using `customer_risk_level` and `customer_recommendations`.
     - For SHAP: Get values with `pipeline['shap_explainer'].shap_values(encoded)`, create DataFrame, plot with `px.bar`.
     - For LIME: Use `explain_customer_lime` and display as DataFrame.
     - Generate CSV: `pd.DataFrame([user_row]).to_csv().encode('utf-8')`.
     - Dependencies: NumPy for SHAP arrays.
     - Pitfalls: Ensure categorical encoding matches (e.g., 'Yes'/'No' to 1/0); test form submission and explainability without crashing.
     - Step: Add after Overview; test prediction logic with sample inputs.

### 4. **Explainability & Model Performance Page**
   - **Feature**: Global insights into the model (feature importance, SHAP summaries, evaluation metrics).
   - **Why**: Covers "Explainability" sections in steps.ipynb and the XGBoost notebook—shows why the model works.
   - **Sub-features**:
     - **XGBoost Feature Importance**: Horizontal bar chart of model feature importances (using `get_feature_importance`).
     - **SHAP Feature Importance**: Bar chart of mean absolute SHAP values across training data (using `get_shap_values`).
     - **Top SHAP Features Table**: Expandable table of top 10 features.
     - **Model Evaluation**: Accuracy metric, classification report table, confusion matrix heatmap (using `evaluate_model` and `px.imshow`).
   - **Outline**:
     - In `render_explainability(pipeline)`, call `get_feature_importance` and plot with `px.bar`.
     - For SHAP: Compute mean abs values from `get_shap_values(pipeline['shap_explainer'], pipeline['train'])`, plot similarly.
     - For evaluation: Use `evaluate_model`, display metrics with `st.metric` and `st.dataframe`.
     - Dependencies: Scikit-learn for metrics.
     - Pitfalls: SHAP computation can be slow—cache or limit to summary; ensure test data is available in pipeline.
     - Step: Add last; run full app to check all pages integrate.

### General Implementation Notes
- **Dependencies**: Ensure `requirements.txt` includes `streamlit`, `pandas`, `plotly`, `numpy`, `xgboost`, `shap`, `lime`, `scikit-learn`. Install via `pip install -r requirements.txt`.
- **Data Flow**: All pages pull from the cached `pipeline` (loaded via `load_pipeline()`), which includes raw data, model, encoders, and explainers.
- **Testing**: After each step, run the app (`streamlit run app.py`) and test the new feature. Use sample data to avoid real-time issues.
- **Edge Cases**: Handle missing data, invalid inputs (e.g., age < 18), and ensure SHAP/LIME don't fail on edge predictions.
- **Step-by-Step Build Order**: 1) Skeleton, 2) Overview, 3) Prediction, 4) Explainability. This way, we validate incrementally and fix issues early (e.g., if prediction fails, it's isolated).
- **Potential Blocks**: If SHAP/LIME crashes, check for version conflicts (e.g., SHAP needs specific XGBoost). If charts don't render, verify Plotly data formats.

This plan ensures we build a complete, runnable app without getting stuck—let me know which feature to tackle first!